# Word Embeddings

## Setup

### Housekeeping (no interaction required)

In [ ]:
%pip install gensim

❗ Please restart the kernel/runtime after installing the package to ensure that all changes take effect.kl

(Google Colab might initiate a restart on its own)

In [ ]:
import os
from pathlib import Path

from IPython.display import clear_output
import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

In [ ]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')

In [ ]:
def confirm(question: str = "Do you want to execute this cell?"):
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

### Setup (Interaction required)

In [ ]:
### ⬇️⬇️⬇️ Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
LOAD_OWN_DATA = True # Set to True if you want to load your own data
### ⬆️⬆️⬆️

<img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=12> The cell below will attempt to connect to your Google Drive.

*Once prompted, give all demanded permissions*

In [ ]:
if IN_COLAB and confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

## Load the data

### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [ ]:
if LOAD_OWN_DATA:
    print("Loading your raw data...", end=" ")
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    raw_df = pd.read_parquet(RAWDATA_PATH)
    print("Done!")

    print("Loading your lemma data..", end=" ")
    LEMMA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.lemmas.parquet"
    lemma_df = pd.read_parquet(LEMMA_PATH)
    print("Done!")

    print("Loading your token data..", end=" ")
    TOKENS_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.tokens.parquet"
    wordform_df = pd.read_parquet(TOKENS_PATH)
    print("Done!")

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

    print(f"Data loaded successfully! {len(raw_df)} rows.")

### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load from example

In [ ]:
if not LOAD_OWN_DATA:
    RAW_URL = f"https://github.com/zentralbibliothek-zuerich/zblab-summerschool-2026/raw/refs/heads/main/data/{CORPUS_NAME}.filtered.parquet"
    print(f"Loading raw data ...", end=" ")
    raw_df = pd.read_parquet(RAW_URL)
    print("Done!")

    LEMMA_URL = f"https://github.com/zentralbibliothek-zuerich/zblab-summerschool-2026/raw/refs/heads/main/data/{CORPUS_NAME}.filtered.lemmas.parquet"
    print(f"Loading lemma data ...", end=" ")
    lemma_df = pd.read_parquet(LEMMA_URL)
    print("Done!")

    TOKENS_URL = f"https://github.com/zentralbibliothek-zuerich/zblab-summerschool-2026/raw/refs/heads/main/data/{CORPUS_NAME}.filtered.tokens.parquet"
    print(f"Loading token data ...", end=" ")
    wordform_df = pd.read_parquet(TOKENS_URL)
    print("Done!")

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

    print(f"Data loaded successfully! {len(raw_df)} rows.")

### Parsing

The library that calculates the Word2Vec model has specific demands for the datatype of the texts.

Loading from parquet format creates `np.ndarray` objects for the sentences and tokens, but the code needs objects of type `list`.

In [ ]:
import numpy as np

def convert_iterable_to_list(obj):
    """Recursively convert nested ndarrays to nested lists."""
    if isinstance(obj, np.ndarray):
        return [convert_iterable_to_list(item) for item in obj]
    else:
        return obj

raw_df['tokens'] = raw_df['tokens'].progress_apply(convert_iterable_to_list)
raw_df['lemmas'] = raw_df['lemmas'].progress_apply(convert_iterable_to_list)
clear_output()

## Word2Vec

### Prepare Vectors

#### Calculate Model

In [ ]:
from gensim.models import Word2Vec

In [ ]:
# Memory-efficient way to iterate over tokenized sentences
class SentenceIterator:
    def __init__(self, df, column):
        self.df = df
        self.column = column

    def __iter__(self):
        for document in tqdm(self.df[self.column], leave=False, desc="Iterating over sentences"):
            for sentence in document:
                yield sentence

In [ ]:
sentences = SentenceIterator(raw_df, 'lemmas')
model = Word2Vec(sentences=sentences, vector_size=300, window=5, min_count=5, workers=4)
clear_output()

#### Extract and save vectors

In [ ]:
wv = model.wv

In [ ]:
from gensim.models import KeyedVectors

if LOAD_OWN_DATA:
    wv = model.wv
    wv.save_word2vec_format(DATA_DIR / f"{CORPUS_NAME}.word_vectors", binary=True)

#### Load precomputed word vectors

In [ ]:
from gensim.models import KeyedVectors
import urllib.request
if LOAD_OWN_DATA:
    wv = KeyedVectors.load_word2vec_format(str(DATA_DIR / f"{CORPUS_NAME}.word_vectors"), binary=True)
else:
    MODEL_URL = f"https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/{CORPUS_NAME}.word_vectors"
    urllib.request.urlretrieve(MODEL_URL, "/tmp/model.word_vectors")
    wv = KeyedVectors.load_word2vec_format("/tmp/model.word_vectors", binary=True)

### Explore the Word Embeddings

#### Most similar words

In [ ]:
wv.most_similar("Not")

In [ ]:
wv.most_similar("Geld")

### Semantic Arithmetics

In [ ]:
wv.most_similar_cosmul(positive=["Armut", "unverschuldet"], negative=["verschuldet"])[:5]

In [ ]:
wv.most_similar_cosmul(positive=["Armensteuer", "fair"], negative=["unfair"])[:10]

In [ ]:
wv.most_similar_cosmul(positive=["Armenpflege", "Familie"], negative=["Staat"])[:5]

### Export Word Vectors for Tensorboard

Export your own model with the code below or download the prepared files from GitHub.

[⬇️ armenpflege.meta.tsv](https://github.com/zentralbibliothek-zuerich/zblab-summerschool-2026/raw/refs/heads/main/data/armenpflege.meta.tsv?download=)
[⬇️ armenpflege.vecs.tsv](https://github.com/zentralbibliothek-zuerich/zblab-summerschool-2026/raw/refs/heads/main/data/armenpflege.vecs.tsv?download=)

In [ ]:
import csv

def wvec2tsv(infile, outf_vecs, outf_meta):
    with open(infile, "r", encoding='utf-8') as vec_file:
        with open(outf_vecs, "w", encoding='utf-8') as vec_tsv:
            with open(outf_meta, "w", encoding='utf-8') as meta_tsv:
                tsv_writer1 = csv.writer(vec_tsv, delimiter='\t')
                tsv_writer2 = csv.writer(meta_tsv, delimiter='\t')
                # Skip the first line containing metadata
                line = vec_file.readline()
                line = vec_file.readline()
                currvec = line.strip().split(' ')
                counter = 0
                pbar = tqdm(total=wv.vectors.shape[0], desc="Converting word vectors to TSV format")
                while line:
                    counter += 1
                    if currvec[0] == "":
                        break
                    tsv_writer2.writerow([currvec[0]])
                    currvec.pop(0)
                    tsv_writer1.writerow(currvec)
                    currvec = vec_file.readline().strip().split(' ')
                    pbar.update(1)
                pbar.close()

wvec2tsv(str(DATA_DIR / f"{CORPUS_NAME}.word_vectors"), str(DATA_DIR / f"{CORPUS_NAME}.vecs.tsv"), str(DATA_DIR / f"{CORPUS_NAME}.meta.tsv"))
clear_output()

Load the two files into the tensorflow embedding projector.

The embedding projector is available at [https://projector.tensorflow.org/](https://projector.tensorflow.org/).